In [ ]:
import pandas as pd
import numpy as np
import os


class RFMSegmentation:

    def __init__(self, filepath):

        self.filepath = filepath

        self.df = None

        os.makedirs("../outputs/segmentation", exist_ok=True)

    #########################################################

    def load_data(self):

        print("=" * 60)
        print("Loading Customer Dataset")
        print("=" * 60)

        self.df = pd.read_csv(
            self.filepath,
            parse_dates=["InvoiceDate"],
            low_memory=False,
            dtype={
                "Invoice": str,
                "StockCode": str
            }
        )

        print(self.df.shape)

    #########################################################

    def build_rfm(self):

        print("\nBuilding RFM Table...\n")

        snapshot = self.df["InvoiceDate"].max() + pd.Timedelta(days=1)

        rfm = (

            self.df

            .groupby("Customer_ID")

            .agg(

                Recency=("InvoiceDate",
                         lambda x: (snapshot - x.max()).days),

                Frequency=("Invoice",
                           "nunique"),

                Monetary=("Total_Price",
                          "sum")

            )

            .reset_index()

        )

        #########################################################

        rfm["R_Score"] = pd.qcut(

            rfm["Recency"],

            5,

            labels=[5,4,3,2,1]

        ).astype(int)

        rfm["F_Score"] = pd.qcut(

            rfm["Frequency"].rank(method="first"),

            5,

            labels=[1,2,3,4,5]

        ).astype(int)

        rfm["M_Score"] = pd.qcut(

            rfm["Monetary"],

            5,

            labels=[1,2,3,4,5]

        ).astype(int)

        #########################################################

        rfm["RFM_Score"] = (

            rfm["R_Score"].astype(str)

            + rfm["F_Score"].astype(str)

            + rfm["M_Score"].astype(str)

        )

        #########################################################

        def segment(row):

            r = row["R_Score"]
            f = row["F_Score"]
            m = row["M_Score"]

            if r >= 4 and f >= 4 and m >= 4:
                return "Champions"

            elif r >= 3 and f >= 4:
                return "Loyal Customers"

            elif r >= 4 and f >= 2:
                return "Potential Loyalists"

            elif r >= 3 and f <= 2:
                return "Need Attention"

            elif r <= 2 and f >= 3:
                return "At Risk"

            else:
                return "Lost"

        rfm["Segment"] = rfm.apply(segment, axis=1)

        self.rfm = rfm

    #########################################################

    def summary(self):

        print()

        print("=" * 50)

        print("Customer Segments")

        print("=" * 50)

        print(

            self.rfm["Segment"]

            .value_counts()

        )

    #########################################################

    def save(self):

        self.rfm.to_csv(

            "../outputs/segmentation/rfm_segmentation.csv",

            index=False

        )

        print("\nRFM Dataset Saved")

    #########################################################

    def run(self):

        self.load_data()

        self.build_rfm()

        self.summary()

        self.save()

        print("\nCustomer Segmentation Completed")


if __name__ == "__main__":

    obj = RFMSegmentation(

        r"C:\Users\ka843\Coding\Amdox Internship_project\data\clean\customer_sales.csv"

    )

    obj.run()